# 階段二：探索性資料分析 (EDA)

本 Notebook 負責：
1. 整體住房率趨勢
2. 各縣市、各星等住房率比較
3. 季節性與假日效應
4. 旅客結構分析
5. 房價與住房率關係
6. 星級認證效益初探
7. 疫後恢復分析

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = ['Microsoft JhengHei', 'DejaVu Sans']
matplotlib.rcParams['axes.unicode_minus'] = False
import seaborn as sns
from pathlib import Path

FIGURES_DIR = Path('..') / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# 載入清理後資料
df = pd.read_csv('../data/processed/hotel_combined.csv', parse_dates=['year_month'])
print(f'資料筆數：{len(df)}，欄位數：{len(df.columns)}')
df.head()

## 1. 整體住房率趨勢

In [ ]:
# 整體月住房率趨勢
if 'year_month' in df.columns and 'occupancy_rate' in df.columns:
    monthly = df.groupby('year_month')['occupancy_rate'].agg(['mean','median','std']).reset_index()
    
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(monthly['year_month'], monthly['mean'], '-o', color='#38bdf8', linewidth=2, markersize=5, label='平均住房率')
    ax.fill_between(monthly['year_month'], monthly['mean']-monthly['std'], monthly['mean']+monthly['std'], alpha=0.2, color='#38bdf8')
    ax.axhline(monthly['mean'].mean(), color='red', linestyle='--', alpha=0.5, label=f'總平均 {monthly["mean"].mean():.1f}%')
    ax.set_xlabel('月份')
    ax.set_ylabel('平均住房率 (%)')
    ax.set_title('台灣星級觀光旅館月住房率趨勢（2023–2025）', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '02_monthly_trend.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # 年度比較
    if 'year' in df.columns:
        yearly = df.groupby('year')['occupancy_rate'].mean()
        print('\n年度平均住房率：')
        for y, v in yearly.items():
            print(f'  {y}：{v:.1f}%')

## 2. 各縣市、各星等住房率比較

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 各縣市住房率
if 'city' in df.columns and 'occupancy_rate' in df.columns:
    city_occ = df.groupby('city')['occupancy_rate'].mean().sort_values(ascending=True)
    city_occ.plot(kind='barh', ax=axes[0], color='#0ea5e9', edgecolor='white')
    axes[0].set_xlabel('平均住房率 (%)')
    axes[0].set_title('各縣市平均住房率', fontsize=12)
    axes[0].axvline(df['occupancy_rate'].mean(), color='red', linestyle='--', alpha=0.5)

# 各星等住房率
if 'star_rating' in df.columns and 'occupancy_rate' in df.columns:
    star_data = df[df['star_rating'].notna()]
    sns.boxplot(data=star_data, x='star_rating', y='occupancy_rate', ax=axes[1], palette='Blues_d')
    axes[1].set_xlabel('星等')
    axes[1].set_ylabel('住房率 (%)')
    axes[1].set_title('各星等住房率分佈', fontsize=12)

plt.tight_layout()
plt.savefig(FIGURES_DIR / '02_city_star_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. 季節性與假日效應

In [ ]:
if 'month' in df.columns and 'occupancy_rate' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # 各月份平均住房率
    month_occ = df.groupby('month')['occupancy_rate'].mean()
    colors = ['#ef4444' if m in [7,8,10] else '#f59e0b' if m in [1,2,4] else '#38bdf8' for m in month_occ.index]
    month_occ.plot(kind='bar', ax=axes[0], color=colors, edgecolor='white')
    axes[0].set_xlabel('月份')
    axes[0].set_ylabel('平均住房率 (%)')
    axes[0].set_title('各月份平均住房率（紅=旺季 黃=假日月 藍=淡季）', fontsize=12)
    axes[0].axhline(df['occupancy_rate'].mean(), color='black', linestyle='--', alpha=0.3)
    axes[0].set_xticklabels(month_occ.index, rotation=0)
    
    # 月份×年份 熱力圖
    if 'year' in df.columns:
        pivot = df.pivot_table(values='occupancy_rate', index='month', columns='year', aggfunc='mean')
        sns.heatmap(pivot, annot=True, fmt='.1f', cmap='Blues', ax=axes[1], cbar_kws={'label': '住房率 (%)'})
        axes[1].set_title('月份×年份 住房率熱力圖', fontsize=12)
        axes[1].set_xlabel('年份')
        axes[1].set_ylabel('月份')
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '02_seasonality.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. 房價與住房率關係

In [ ]:
if 'avg_price' in df.columns and 'occupancy_rate' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # 房價 vs 住房率 散佈圖
    axes[0].scatter(df['avg_price'], df['occupancy_rate'], alpha=0.3, color='#38bdf8', s=20)
    z = np.polyfit(df['avg_price'].dropna(), df.loc[df['avg_price'].notna(), 'occupancy_rate'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df['avg_price'].min(), df['avg_price'].max(), 100)
    axes[0].plot(x_line, p(x_line), 'r--', linewidth=2, label=f'趨勢線 (r={df["avg_price"].corr(df["occupancy_rate"]):.3f})')
    axes[0].set_xlabel('平均房價 (NT$)')
    axes[0].set_ylabel('住房率 (%)')
    axes[0].set_title('房價 vs 住房率', fontsize=12)
    axes[0].legend()
    
    # RevPAR
    df['revpar'] = df['avg_price'] * df['occupancy_rate'] / 100
    if 'star_rating' in df.columns:
        revpar_star = df[df['star_rating'].notna()].groupby('star_rating')['revpar'].mean().sort_values(ascending=True)
        revpar_star.plot(kind='barh', ax=axes[1], color='#0ea5e9', edgecolor='white')
        axes[1].set_xlabel('RevPAR (NT$)')
        axes[1].set_title('各星等平均 RevPAR（每可售房收入）', fontsize=12)
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '02_price_occupancy.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5. 星級認證效益初探

比較有星等與無星等觀光旅館的住房率、房價、RevPAR 差異

In [ ]:
# 星級認證效益分析
if 'star_rating' in df.columns or 'star_rank' in df.columns:
    rank_col = 'star_rank' if 'star_rank' in df.columns else 'star_rating'
    df['has_star'] = df[rank_col].notna().map({True: '有星等', False: '無星等'})
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # 住房率比較
    if 'occupancy_rate' in df.columns:
        df.boxplot(column='occupancy_rate', by='has_star', ax=axes[0])
        axes[0].set_title('住房率比較', fontsize=12)
        axes[0].set_xlabel('')
        axes[0].set_ylabel('住房率 (%)')
    
    # 房價比較
    if 'avg_price' in df.columns:
        df.boxplot(column='avg_price', by='has_star', ax=axes[1])
        axes[1].set_title('平均房價比較', fontsize=12)
        axes[1].set_xlabel('')
        axes[1].set_ylabel('平均房價 (NT$)')
    
    # RevPAR 比較
    if 'revpar' in df.columns:
        df.boxplot(column='revpar', by='has_star', ax=axes[2])
        axes[2].set_title('RevPAR 比較', fontsize=12)
        axes[2].set_xlabel('')
        axes[2].set_ylabel('RevPAR (NT$)')
    
    plt.suptitle('有星等 vs 無星等觀光旅館經營效益比較', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '02_star_certification.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # 數字比較
    print('\n=== 有星等 vs 無星等 統計 ===')
    compare_cols = [c for c in ['occupancy_rate', 'avg_price', 'revpar'] if c in df.columns]
    print(df.groupby('has_star')[compare_cols].mean().round(2))
else:
    print('無星等欄位，略過此分析')

## 6. 疫後恢復分析（2023→2025）

In [ ]:
# 疫後恢復分析
if 'year' in df.columns and 'occupancy_rate' in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # 年度住房率變化
    yearly_occ = df.groupby('year')['occupancy_rate'].mean()
    yearly_occ.plot(kind='bar', ax=axes[0], color=['#64748b', '#0ea5e9', '#22c55e'], edgecolor='white')
    axes[0].set_xlabel('年份')
    axes[0].set_ylabel('平均住房率 (%)')
    axes[0].set_title('年度平均住房率變化', fontsize=12)
    axes[0].set_xticklabels(yearly_occ.index, rotation=0)
    for i, v in enumerate(yearly_occ):
        axes[0].text(i, v+0.5, f'{v:.1f}%', ha='center', fontweight='bold')
    
    # 國際旅客比例趨勢
    if 'international_guests' in df.columns and 'domestic_guests' in df.columns:
        total = df['domestic_guests'] + df['international_guests']
        df['intl_ratio_calc'] = np.where(total > 0, df['international_guests'] / total * 100, np.nan)
        yearly_intl = df.groupby('year')['intl_ratio_calc'].mean()
        yearly_intl.plot(kind='bar', ax=axes[1], color=['#64748b', '#0ea5e9', '#22c55e'], edgecolor='white')
        axes[1].set_xlabel('年份')
        axes[1].set_ylabel('國際旅客比例 (%)')
        axes[1].set_title('國際旅客比例恢復趨勢', fontsize=12)
        axes[1].set_xticklabels(yearly_intl.index, rotation=0)
        for i, v in enumerate(yearly_intl):
            axes[1].text(i, v+0.3, f'{v:.1f}%', ha='center', fontweight='bold')
    
    plt.suptitle('疫後恢復分析（2023→2025）', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '02_recovery_analysis.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. 相關係數矩陣與 EDA 摘要

In [ ]:
# 相關係數矩陣
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if len(numeric_cols) > 3:
    corr = df[numeric_cols].corr()
    
    fig, ax = plt.subplots(figsize=(12, 10))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                ax=ax, square=True, linewidths=0.5, vmin=-1, vmax=1)
    ax.set_title('數值欄位相關係數矩陣', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / '02_correlation_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    # 與住房率相關性最高的欄位
    if 'occupancy_rate' in corr.columns:
        occ_corr = corr['occupancy_rate'].drop('occupancy_rate').abs().sort_values(ascending=False)
        print('\n=== 與住房率相關性最高的欄位 ===')
        for feat, val in occ_corr.head(10).items():
            print(f'  {feat}: {val:.3f}')

print('\n\n✅ EDA 完成！下一步：執行 03_feature_engineering.ipynb 進行特徵工程')